# 16. The Storage Location Assignment Problem (SLAP)
## Tier 1: The Pen & Paper Method - Set Covering Mathematical Formulation

### Key assumptions
- Each product must be assigned to exactly one storage location
- Storage locations have limited capacity
- Assignment costs depend on product velocity and location distance
- Product affinities influence assignment decisions

### Approach (step-by-step)
1. **Problem Formulation**: Define sets, parameters, and decision variables
2. **Objective Function**: Minimize total assignment cost including affinity penalties
3. **Constraints**: Ensure capacity limits and exact assignment requirements
4. **Solution Method**: Use mixed-integer programming for exact optimality
5. **Analysis**: Sensitivity analysis and result interpretation

### What to look for in the results
- Optimal assignment matrix showing which product goes to which location
- Total cost breakdown (assignment costs + affinity penalties)
- Capacity utilization for each storage location
- Sensitivity analysis showing how changes affect the optimal solution

### Concrete example (from the source)
We'll implement the example with 3 products and 3 locations:
- Products with velocities: v₁=50, v₂=25, v₃=10
- Locations with distances: d_A=2.5, d_B=4.2, d_C=6.8
- Expected optimal assignment: Product 1→Location A, Product 2→Location B, Product 3→Location C
- Expected total cost: 298

In [1]:
from typing import Tuple, List, Dict, Set
from itertools import product

# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class Product:
    """Represents a product/SKU in the warehouse"""
    id: int
    velocity: float  # Demand frequency
    space_req: float  # Space requirement
    name: str = ""

@dataclass
class Location:
    """Represents a storage location"""
    id: str
    distance: float  # Distance from I/O point
    capacity: float  # Storage capacity

@dataclass
class SLAPInstance:
    """Storage Location Assignment Problem instance"""
    products: List[Product]
    locations: List[Location]
    affinity_matrix: np.ndarray  # Product affinity scores
    alpha: float = 1.0  # Affinity penalty weight
    beta: float = 0.5  # Affinity penalty coefficient

In [ ]:
def create_cost_matrix(instance: SLAPInstance) -> np.ndarray:
    """
    Calculate the cost matrix for product-location assignments.
    Cost = distance * velocity + affinity penalty
    """
    n_products = len(instance.products)
    n_locations = len(instance.locations)
    
    cost_matrix = np.zeros((n_products, n_locations))
    
    for i, product in enumerate(instance.products):
        for j, location in enumerate(instance.locations):
            # Base cost: distance * velocity
            base_cost = location.distance * product.velocity
            
            # Affinity penalty: beta * sum(affinity * distance_penalty)
            affinity_penalty = 0
            for k in range(n_products):
                if k != i:
                    # Penalty for separating related products
                    affinity_penalty += (instance.beta * 
                                        instance.affinity_matrix[i, k] * 
                                        location.distance)
            
            cost_matrix[i, j] = base_cost + affinity_penalty
    
    return cost_matrix

def solve_slap_exact(instance: SLAPInstance) -> Tuple[np.ndarray, float, Dict]:
    """
    Solve SLAP using exact enumeration (for small instances).
    Returns assignment matrix, total cost, and solution details.
    """
    n_products = len(instance.products)
    n_locations = len(instance.locations)
    
    cost_matrix = create_cost_matrix(instance)
    
    # Generate all possible assignments (each product to one location)
    best_assignment = None
    best_cost = float('inf')
    best_details = {}
    
    # For small instances, we can enumerate all possibilities
    from itertools import product
    
    for assignment in product(range(n_locations), repeat=n_products):
        # Check capacity constraints
        location_loads = np.zeros(n_locations)
        feasible = True
        
        for i, loc_idx in enumerate(assignment):
            location_loads[loc_idx] += instance.products[i].space_req
            if location_loads[loc_idx] > instance.locations[loc_idx].capacity:
                feasible = False
                break
        
        if not feasible:
            continue
        
        # Calculate total cost
        total_cost = sum(cost_matrix[i, assignment[i]] for i in range(n_products))
        
        if total_cost < best_cost:
            best_cost = total_cost
            best_assignment = assignment
            best_details = {
                'assignment': assignment,
                'location_loads': location_loads,
                'cost_matrix': cost_matrix.copy()
            }
    
    # Convert assignment to matrix format
    assignment_matrix = np.zeros((n_products, n_locations))
    if best_assignment:
        for i, loc_idx in enumerate(best_assignment):
            assignment_matrix[i, loc_idx] = 1
    
    return assignment_matrix, best_cost, best_details

In [ ]:
# Create the concrete example from the source
def create_concrete_example():
    """Create the example with 3 products and 3 locations"""
    # Products with velocities: v₁=50, v₂=25, v₃=10
    products = [
        Product(id=0, velocity=50, space_req=1.0, name="Product 1"),
        Product(id=1, velocity=25, space_req=1.0, name="Product 2"),
        Product(id=2, velocity=10, space_req=1.0, name="Product 3")
    ]
    
    # Locations with distances: d_A=2.5, d_B=4.2, d_C=6.8
    locations = [
        Location(id='A', distance=2.5, capacity=2.0),
        Location(id='B', distance=4.2, capacity=2.0),
        Location(id='C', distance=6.8, capacity=2.0)
    ]
    
    # Affinity matrix (simplified for this example)
    affinity_matrix = np.array([
        [0, 8, 5],  # Product 1 affinities
        [8, 0, 3],  # Product 2 affinities
        [5, 3, 0]   # Product 3 affinities
    ])
    
    return SLAPInstance(
        products=products,
        locations=locations,
        affinity_matrix=affinity_matrix,
        alpha=1.0,
        beta=0.5
    )

# Create and solve the instance
instance = create_concrete_example()
assignment_matrix, total_cost, details = solve_slap_exact(instance)

print("=== SLAP Set Covering Formulation Results ===")
print(f"\nTotal Cost: {total_cost:.2f}")
print(f"\nCost Matrix:")
cost_df = pd.DataFrame(
    details['cost_matrix'],
    index=[f"Product {i+1}" for i in range(len(instance.products))],
    columns=[f"Location {loc.id}" for loc in instance.locations]
)
print(cost_df.round(2))

print(f"\nOptimal Assignment:")
assignment_df = pd.DataFrame(
    assignment_matrix,
    index=[f"Product {i+1}" for i in range(len(instance.products))],
    columns=[f"Location {loc.id}" for loc in instance.locations]
)
print(assignment_df.astype(int))

print(f"\nLocation Utilization:")
for i, loc in enumerate(instance.locations):
    load = details['location_loads'][i]
    utilization = (load / loc.capacity) * 100
    print(f"Location {loc.id}: {load:.1f}/{loc.capacity:.1f} ({utilization:.1f}%)")

=== SLAP Set Covering Formulation Results ===

Total Cost: 276.30

Cost Matrix:
           Location A  Location B  Location C
Product 1      141.25       237.3       384.2
Product 2       76.25       128.1       207.4
Product 3       35.00        58.8        95.2

Optimal Assignment:
           Location A  Location B  Location C
Product 1           1           0           0
Product 2           1           0           0
Product 3           0           1           0

Location Utilization:
Location A: 2.0/2.0 (100.0%)
Location B: 1.0/2.0 (50.0%)
Location C: 0.0/2.0 (0.0%)


In [ ]:
try:
    # Visualization of results
    def visualize_slap_solution(instance: SLAPInstance, assignment_matrix: np.ndarray, 
                               cost_matrix: np.ndarray, total_cost: float):
        """Create comprehensive visualization of SLAP solution"""
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        fig.suptitle('SLAP Set Covering Formulation - Solution Analysis', fontsize=16, fontweight='bold')
        
        # 1. Cost Matrix Heatmap
        ax1 = axes[0, 0]
        sns.heatmap(cost_matrix, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax1,
                    xticklabels=[f"Loc {loc.id}" for loc in instance.locations],
                    yticklabels=[f"Prod {i+1}" for i in range(len(instance.products))])
        ax1.set_title('Cost Matrix (Assignment Costs)')
        ax1.set_xlabel('Storage Locations')
        ax1.set_ylabel('Products')
        
        # 2. Assignment Visualization
        ax2 = axes[0, 1]
        assignment_df = pd.DataFrame(
            assignment_matrix,
            index=[f"P{i+1}" for i in range(len(instance.products))],
            columns=[f"L{loc.id}" for loc in instance.locations]
        )
        sns.heatmap(assignment_df, annot=True, fmt='d', cmap='Blues', ax=ax2)
        ax2.set_title('Optimal Assignment Matrix')
        ax2.set_xlabel('Storage Locations')
        ax2.set_ylabel('Products')
        
        # 3. Cost Breakdown
        ax3 = axes[1, 0]
        product_costs = []
        for i in range(len(instance.products)):
            for j in range(len(instance.locations)):
                if assignment_matrix[i, j] == 1:
                    product_costs.append(cost_matrix[i, j])
        
        colors = plt.cm.Set3(np.linspace(0, 1, len(product_costs)))
        bars = ax3.bar([f'P{i+1}' for i in range(len(product_costs))], 
                       product_costs, color=colors)
        ax3.set_title('Cost Breakdown by Product')
        ax3.set_xlabel('Product')
        ax3.set_ylabel('Assignment Cost')
        ax3.grid(True, alpha=0.3)
        
        # Add value labels on bars
        for bar, cost in zip(bars, product_costs):
            height = bar.get_height()
            ax3.text(bar.get_x() + bar.get_width()/2., height + 1,
                    f'{cost:.1f}', ha='center', va='bottom')
        
        # 4. Location Utilization
        ax4 = axes[1, 1]
        location_loads = np.sum(assignment_matrix * 
                               np.array([p.space_req for p in instance.products]).reshape(-1, 1), axis=0)
        
        capacities = [loc.capacity for loc in instance.locations]
        utilizations = (location_loads / capacities) * 100
        
        x_pos = np.arange(len(instance.locations))
        bars = ax4.bar(x_pos, utilizations, color=['skyblue', 'lightgreen', 'salmon'])
        ax4.set_title('Location Utilization')
        ax4.set_xlabel('Storage Location')
        ax4.set_ylabel('Utilization (%)')
        ax4.set_xticks(x_pos)
        ax4.set_xticklabels([f'Loc {loc.id}' for loc in instance.locations])
        ax4.grid(True, alpha=0.3)
        
        # Add utilization percentage labels
        for bar, util in zip(bars, utilizations):
            height = bar.get_height()
            ax4.text(bar.get_x() + bar.get_width()/2., height + 1,
                    f'{util:.1f}%', ha='center', va='bottom')
        
        plt.tight_layout()
        plt.show()
        
        # Print summary
        print(f"\n{'='*50}")
        print(f"SOLUTION SUMMARY")
        print(f"{'='*50}")
        print(f"Total Optimal Cost: {total_cost:.2f}")
        print(f"\nOptimal Assignment:")
        for i in range(len(instance.products)):
            for j in range(len(instance.locations)):
                if assignment_matrix[i, j] == 1:
                    print(f"  Product {i+1} → Location {instance.locations[j].id} (Cost: {cost_matrix[i,j]:.2f})")
        print(f"\nLocation Utilization:")
        for j, loc in enumerate(instance.locations):
            print(f"  Location {loc.id}: {location_loads[j]:.1f}/{loc.capacity:.1f} ({utilizations[j]:.1f}%)")
    
    # Visualize the solution
    visualize_slap_solution(instance, assignment_matrix, details['cost_matrix'], total_cost)
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: Unknown format code 'd' for object of type 'float'...]

In [ ]:
# Sensitivity Analysis
def sensitivity_analysis(instance: SLAPInstance):
    """Perform sensitivity analysis on key parameters"""
    
    print("\n" + "="*60)
    print("SENSITIVITY ANALYSIS")
    print("="*60)
    
    # 1. Vary beta (affinity penalty coefficient)
    beta_values = [0.0, 0.25, 0.5, 0.75, 1.0]
    costs_by_beta = []
    
    print("\n1. Impact of Affinity Penalty Coefficient (β):")
    print(f"{'β':<6} {'Total Cost':<12} {'Assignment Change':<15}")
    print("-" * 35)
    
    for beta in beta_values:
        test_instance = SLAPInstance(
            products=instance.products,
            locations=instance.locations,
            affinity_matrix=instance.affinity_matrix,
            alpha=instance.alpha,
            beta=beta
        )
        _, cost, _ = solve_slap_exact(test_instance)
        costs_by_beta.append(cost)
        print(f"{beta:<6.2f} {cost:<12.2f} {'-':<15}")
    
    # 2. Vary location distances
    distance_multipliers = [0.8, 0.9, 1.0, 1.1, 1.2]
    costs_by_distance = []
    
    print("\n2. Impact of Distance Multiplier:")
    print(f"{'Mult.':<8} {'Total Cost':<12} {'Change %':<10}")
    print("-" * 32)
    
    base_cost = total_cost
    for mult in distance_multipliers:
        modified_locations = [
            Location(loc.id, loc.distance * mult, loc.capacity)
            for loc in instance.locations
        ]
        test_instance = SLAPInstance(
            products=instance.products,
            locations=modified_locations,
            affinity_matrix=instance.affinity_matrix,
            alpha=instance.alpha,
            beta=instance.beta
        )
        _, cost, _ = solve_slap_exact(test_instance)
        costs_by_distance.append(cost)
        change_pct = ((cost - base_cost) / base_cost) * 100
        print(f"{mult:<8.2f} {cost:<12.2f} {change_pct:<10.1f}%")
    
    # 3. Vary product velocities
    velocity_multipliers = [0.8, 0.9, 1.0, 1.1, 1.2]
    costs_by_velocity = []
    
    print("\n3. Impact of Velocity Multiplier:")
    print(f"{'Mult.':<8} {'Total Cost':<12} {'Change %':<10}")
    print("-" * 32)
    
    for mult in velocity_multipliers:
        modified_products = [
            Product(prod.id, prod.velocity * mult, prod.space_req, prod.name)
            for prod in instance.products
        ]
        test_instance = SLAPInstance(
            products=modified_products,
            locations=instance.locations,
            affinity_matrix=instance.affinity_matrix,
            alpha=instance.alpha,
            beta=instance.beta
        )
        _, cost, _ = solve_slap_exact(test_instance)
        costs_by_velocity.append(cost)
        change_pct = ((cost - base_cost) / base_cost) * 100
        print(f"{mult:<8.2f} {cost:<12.2f} {change_pct:<10.1f}%")
    
    # Create sensitivity plots
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('SLAP Sensitivity Analysis', fontsize=16, fontweight='bold')
    
    # Beta sensitivity
    axes[0].plot(beta_values, costs_by_beta, 'bo-', linewidth=2, markersize=8)
    axes[0].set_title('Impact of Affinity Penalty (β)')
    axes[0].set_xlabel('Affinity Penalty Coefficient (β)')
    axes[0].set_ylabel('Total Cost')
    axes[0].grid(True, alpha=0.3)
    
    # Distance sensitivity
    axes[1].plot(distance_multipliers, costs_by_distance, 'ro-', linewidth=2, markersize=8)
    axes[1].set_title('Impact of Distance Multiplier')
    axes[1].set_xlabel('Distance Multiplier')
    axes[1].set_ylabel('Total Cost')
    axes[1].grid(True, alpha=0.3)
    
    # Velocity sensitivity
    axes[2].plot(velocity_multipliers, costs_by_velocity, 'go-', linewidth=2, markersize=8)
    axes[2].set_title('Impact of Velocity Multiplier')
    axes[2].set_xlabel('Velocity Multiplier')
    axes[2].set_ylabel('Total Cost')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Perform sensitivity analysis
sensitivity_analysis(instance)


SENSITIVITY ANALYSIS

1. Impact of Affinity Penalty Coefficient (β):
β      Total Cost   Assignment Change
-----------------------------------
0.00   229.50       -              
0.25   252.90       -              
0.50   276.30       -              
0.75   299.70       -              
1.00   323.10       -              

2. Impact of Distance Multiplier:
Mult.    Total Cost   Change %  
--------------------------------
0.80     221.04       -20.0     %
0.90     248.67       -10.0     %
1.00     276.30       0.0       %
1.10     303.93       10.0      %
1.20     331.56       20.0      %

3. Impact of Velocity Multiplier:
Mult.    Total Cost   Change %  
--------------------------------
0.80     230.40       -16.6     %
0.90     253.35       -8.3      %
1.00     276.30       0.0       %
1.10     299.25       8.3       %
1.20     322.20       16.6      %
Starting Priority Queue Heuristic...
Initial priority queue: [(5, 313.85388407548226), (4, 442.053446976681), (3, 477.6134162777644), 

### Why this Tier exists vs earlier Tiers
This is Tier 1, the foundational mathematical formulation that provides:
- **Exact optimality** guarantees through mathematical programming
- **Provably optimal solutions** for small instances
- **Baseline for comparison** with heuristic and metaheuristic methods
- **Mathematical understanding** of the problem structure and trade-offs

### Pros / Cons vs other approaches
**Pros:**
- Guarantees optimal solution
- Provides mathematical proof of optimality
- Clear problem formulation and constraints
- Useful for theoretical analysis and sensitivity studies

**Cons:**
- Computationally expensive for large instances
- Not scalable to real-world warehouse sizes
- Requires specialized optimization solvers
- Limited flexibility for dynamic changes

### When to use this Tier
- **Small warehouses** with limited products and locations
- **Benchmarking** other solution methods
- **Academic research** and theoretical analysis
- **Strategic planning** where optimality is critical
- **Teaching** optimization concepts and mathematical formulation